In [1]:
%load_ext autoreload
%autoreload 2
%load_ext dotenv
%dotenv

In [2]:
import os

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"  # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
from pkgimp import *
from bson import ObjectId
from tqdm import tqdm
import time

from nb2p import database, fileop, config, astparse
from nb2p.notebook import Notebook

from prompt import make_llm_prompt, make_llm_prompt_cot

from transformers import AutoModelForCausalLM, AutoTokenizer

/home/haotian/anaconda3/envs/llm-2410/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
DATASET_NAME = 'distilkaggle'

## Load Dataset

In [5]:
ground_truth = fileop.read_json(os.path.join(DATASET_NAME, f"{DATASET_NAME}_groundtruth.json"))
ground_truth[20]

{'func_defs': ['def findGCD(seq):\n    gcd = seq[0]\n    for i in range(1,len(seq)):\n        gcd=math.gcd(gcd, seq[i])\n    return gcd',
  'def findSignature(seq):\n    nonzero_seq = [d for d in seq if d!=0]\n    if len(nonzero_seq)==0:\n        return seq\n    sign = 1 if nonzero_seq[0]>0 else -1\n    gcd = findGCD(seq)\n    return [sign*x//gcd for x in seq]',
  'def findDerivative(seq):\n    return [0] if len(seq)<=1 else [seq[i]-seq[i-1] for i in range(1,len(seq))]',
  "def addAll(seq, node, list):\n    if 'value' in node:\n        list.append( ( seq, node['value'] ) )\n    for key in node:\n        if key != 'value':\n            addAll(seq + [key], node[key], list)",
  'def findNext(seq, trie):\n    while True:\n        nonZeroIndex=-1\n        for i in range(0,len(seq)):\n            if seq[i]!=0:\n                nonZeroIndex=i\n                break\n        if nonZeroIndex<0:\n            return 0\n        signature=findSignature(seq)\n        list=trie.prefix( signature )\n 

## Make LLM Prompt

In [6]:
PROMPT_FUNC = make_llm_prompt

In [7]:
test_prompt = PROMPT_FUNC(ground_truth[20])
print(test_prompt)

Suppose you have a data science notebook and want to extract pipeline components based on their semantic purposes. There are two requirements. First, each component should contain consecutive code, one more more lines, in the notebook. You should output one code cell for each component. You cannot modify, swap, or exclude any code. Second, each component should represent a specific stage in the data science process. Components can have the same stage. The example stages are:
- data acquisition (such as load, collect, obtain, capture, survey)
- data preparation (such as explore, wrangle, clean, filter, organize)
- storage (such as preserve, archive, warehouse, log, recycle)
- feature engineering (such as feature, label, annotate)
- modeling (such as classify, cluster, mine, analyze, process)
- training (such as tune, optimize)
- evaluation (such as validate, test, verify, review)
- prediction (such as discover, derive, determine)
- interpretation (such as transform, visualize, render, t

In [8]:
MODEL_NAME = "Qwen2.5-Coder-7B-Instruct"
# MODEL_NAME = "DeepSeek-Coder-V2-Lite-Instruct"

MODEL_PATH = os.path.join("/ssd/haotian/scs/llm", MODEL_NAME)

In [9]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype="auto",
    device_map="auto",
    local_files_only=True,
    trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

Loading checkpoint shards: 100%|█████████████████████████████████████████| 4/4 [00:12<00:00,  3.14s/it]


In [10]:
def chat(prompt: str):
    messages = [
        {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], 
                             return_tensors="pt", 
                             truncation=True, 
                             max_length=4096).to(model.device)
    
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=4096
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return response

In [11]:
result = chat(test_prompt)
print(result)

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.


 Based on the stages provided and the code segments in the notebook, we can identify the following pipeline components:

### COMPONENT 1: Data Acquisition
This component involves loading or collecting data from external sources. From the notebook, we see:
```python
train_df= pd.read_csv('../input/train.csv', index_col="Id", nrows=100)
test_df = pd.read_csv('../input/test.csv', index_col="Id", nrows=100)
```
This code reads data from CSV files into pandas DataFrames.

### COMPONENT 2: Data Preparation
This component involves exploring, wrangling, cleaning, and organizing the data. From the notebook, we see:
```python
train_df= train_df['Sequence'].to_dict()
test_df= test_df['Sequence'].to_dict()
seqs={0: [1 for x in range(0,400)]}
for key in train_df:
    seq=train_df[key]
    seq=[int(x) for x in seq.split(',')]
    seqs[key]=seq
for key in test_df:
    seq=test_df[key]
    seq=[int(x) for x in seq.split(',')]
    seqs[key]=seq
```
This code processes the data into a dictionary format,

In [12]:
OUT_DIR = os.path.join(DATASET_NAME, MODEL_NAME, "raw_cot")

os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
for i, gt in tqdm(enumerate(ground_truth), total=len(ground_truth)):
    out_path = os.path.join(OUT_DIR, f"{i}.json")
    if os.path.exists(out_path):
        continue
    
    start_time = time.time() 
    
    try:
        result = chat(PROMPT_FUNC(gt))
    except Exception as e:
        print(f"ERROR {i}: {e}")
    
    end_time = time.time()
    fileop.write_json({"response": result, "time": end_time - start_time}, out_path)

 10%|█████▌                                                   | 101/1024 [2:08:21<35:09:16, 137.11s/it]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
